# Real-Time Fraud Detection: Model Comparison & Benchmark

## 1. Objective & Model Taxonomy
We evaluate a standard progression of machine learning algorithms:
- **Linear Baseline**: Logistic Regression with balanced inverse-frequency class weights
- **Bagging Ensemble**: Random Forest Classifier
- **Gradient Boosting**: LightGBM and XGBoost with loss scaling
- **Unsupervised Anomaly Detector**: Isolation Forest

Evaluation emphasizes **PR-AUC (Precision-Recall AUC)**, which is sensitive to minority positive performance and immune to the false sense of security provided by high TN in ROC-AUC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, roc_curve, average_precision_score, roc_auc_score

from src.data.loader import load_credit_card_data
from src.data.split import time_based_split
from src.features.engineer import TransactionFeatureEngineer
from src.models.baseline import build_logistic_regression_baseline
from src.models.trees import build_random_forest_model, build_lightgbm_model, build_xgboost_model
from src.models.evaluator import evaluate_predictions

# 1. Load data & Leakage-free temporal split
df = load_credit_card_data('../data/raw/creditcard.csv')
train_df, val_df, test_df = time_based_split(df, val_size=0.15, test_size=0.15)

# 2. Feature engineering fitted ONLY on train
fe = TransactionFeatureEngineer()
fe.fit(train_df.drop(columns=['Class']))

X_train = fe.transform(train_df.drop(columns=['Class']))
y_train = train_df['Class'].values
X_val = fe.transform(val_df.drop(columns=['Class']))
y_val = val_df['Class'].values
X_test = fe.transform(test_df.drop(columns=['Class']))
y_test = test_df['Class'].values

In [ ]:
models = {
    'Logistic Regression': build_logistic_regression_baseline(class_weight='balanced'),
    'Random Forest': build_random_forest_model(n_estimators=100, max_depth=10),
    'LightGBM': build_lightgbm_model(n_estimators=100, max_depth=5),
    'XGBoost': build_xgboost_model(n_estimators=100, max_depth=5)
}

val_predictions = {}
metrics_list = []

for name, model in models.items():
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_val)[:, 1]
    val_predictions[name] = probs
    m = evaluate_predictions(y_val, probs, threshold=0.5)
    m['Model'] = name
    metrics_list.append(m)

df_comparison = pd.DataFrame(metrics_list)[['Model', 'pr_auc', 'roc_auc', 'recall', 'precision', 'f1', 'total_cost']]
print("=== Model Comparison Summary Table ===")
display(df_comparison)

## 2. Precision-Recall Curves & ROC Curves
Visualizing the Precision-Recall curves. Notice how PR curves highlight the true operating trade-off under extreme imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Precision-Recall Curve
for name, probs in val_predictions.items():
    precision, recall, _ = precision_recall_curve(y_val, probs)
    pr_auc = average_precision_score(y_val, probs)
    axes[0].plot(recall, precision, lw=2, label=f'{name} (PR-AUC = {pr_auc:.3f})')

axes[0].set_xlabel('Recall (Fraud Detection Rate)', fontsize=11)
axes[0].set_ylabel('Precision (Positive Predictive Value)', fontsize=11)
axes[0].set_title('Precision-Recall Curves (Validation Set)', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower left')
axes[0].grid(True, alpha=0.3)

# 2. ROC Curve
for name, probs in val_predictions.items():
    fpr, tpr, _ = roc_curve(y_val, probs)
    roc_auc = roc_auc_score(y_val, probs)
    axes[1].plot(fpr, tpr, lw=2, label=f'{name} (ROC-AUC = {roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('Receiver Operating Characteristic (ROC)', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()